# SciGeoGuard-X v11 — EarthRefine Novelty Experiments
## Claim-conditioned refinement obligations, evidence certificates, and counterexample-guided repair

This notebook tests the **next novelty layer** proposed for SciGeoGuard-X. It is designed to answer five research questions:

1. **RQ1 — Refinement obligations:** Can EO scientific constraints be represented as named, inspectable obligations over typed `Artifact`, `Step`, and `Intent` records?
2. **RQ2 — Counterexample certificates:** Can each `REFUTED` or `UNPROVEN` verdict be converted into a machine-readable certificate containing the violated predicate, evidence, missing evidence, and repairability status?
3. **RQ3 — Minimal repair:** Can a counterexample-guided search find the smallest **safe** workflow edit that turns a refuted workflow into `PROVEN` without inventing scientific evidence?
4. **RQ4 — Metamorphic testing:** Starting from valid workflows, do controlled scientific-semantic mutations become `REFUTED`, and can repair recover the original scientific validity?
5. **RQ5 — Benchmark repairability:** On the frozen 180-case subset, what fraction of refuted workflows admits a safe repair under the current repair library?

### Evidence boundary
The 180-case reference is still a **generated scientific-invariant reference**, not independent human ground truth. Repair success on it measures regression-level repairability, not real-world accuracy. The metamorphic cases in this notebook are author-defined controlled tests. Do **not** present either as independent expert validation.


## 0. Setup
The notebook has no paid API dependency. It uses only Python, pandas, NumPy and matplotlib. An optional cell later demonstrates how the same predicates can be encoded in Z3; the core experiments do not depend on Z3 so the notebook remains runnable offline.


In [ ]:
from pathlib import Path
import json, math, shutil, sys, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 20261005
np.random.seed(SEED)

# Works in Colab, Jupyter, or an extracted local package.
RUN_DIR = Path('/content/scigeoguard_v11_results') if Path('/content').exists() else Path.cwd() / 'scigeoguard_v11_results'
RUN_DIR.mkdir(parents=True, exist_ok=True)
print('RUN_DIR =', RUN_DIR)


## 1. Frozen verifier + new EarthRefine layer
The block below contains the frozen SciGeoGuard-X verifier semantics used in the existing experiments, then adds:

- a named **obligation library** (`R-*` identifiers),
- `EvidenceCertificate-v0.1`,
- a conservative repair-candidate generator,
- breadth-first **minimal repair synthesis**,
- a ten-family scientific metamorphic suite.

The repair synthesizer deliberately refuses several tempting edits. It does **not** invent provenance, fabricate cloud-mask status, change a vertical datum label, widen a temporal tolerance, or overwrite physical observations just to obtain a pass.


In [ ]:
from dataclasses import dataclass, field, asdict, replace
from datetime import datetime, timezone, timedelta
from typing import Any, Dict, List, Optional
from copy import deepcopy
import json, hashlib, math
from collections import deque

@dataclass
class Artifact:
    name: str
    quantity: str
    unit: Optional[str] = None
    scale: str = 'continuous'
    crs: Optional[str] = None
    resolution_m: Optional[float] = None
    temporal_resolution_h: Optional[float] = None
    processing_level: Optional[str] = None
    surface_model: Optional[str] = None
    vertical_datum: Optional[str] = None
    acquisition_time: Optional[str] = None
    available_time: Optional[str] = None
    provenance: Optional[str] = None
    band_role: Optional[str] = None
    nodata_semantics: Optional[str] = None
    uncertainty: Optional[float] = None
    allowed_values: Optional[List[float]] = None
    observed_min: Optional[float] = None
    observed_max: Optional[float] = None
    attrs: Dict[str, Any] = field(default_factory=dict)

@dataclass
class Step:
    op: str
    params: Dict[str, Any] = field(default_factory=dict)

@dataclass
class Intent:
    goal: str
    required_quantity: Optional[str] = None
    required_unit: Optional[str] = None
    decision_time: Optional[str] = None
    max_resolution_m: Optional[float] = None
    requires_bare_earth: bool = False
    requires_cloud_mask: bool = False
    max_uncertainty: Optional[float] = None
    temporal_tolerance_h: Optional[float] = None
    required_value_range: Optional[List[float]] = None

AREA_UNITS = {'m2','km2','ha'}
LENGTH_UNITS = {'m','km','cm','mm'}
RATE_UNITS = {'mm/h','mm/hr','MM/Hr'}
CATEGORICAL_SCALES = {'categorical','nominal','ordinal','binary'}
CONTINUOUS_INTERPOLATORS = {'bilinear','cubic','lanczos'}
UNIT_DIMENSION = {'m2':'area','km2':'area','ha':'area','m':'length','km':'length','cm':'length','mm':'length_or_accum','mm/h':'rate','mm/hr':'rate','MM/Hr':'rate','dimensionless':'dimensionless','1':'dimensionless','K':'temperature','degC':'temperature'}

def parse_dt(x):
    if not x: return None
    d=datetime.fromisoformat(str(x).replace('Z','+00:00'))
    if d.tzinfo is None: d=d.replace(tzinfo=timezone.utc)
    return d

def unit_dimension(u): return UNIT_DIMENSION.get(u)
def artifact_copy(x): return replace(x, attrs=dict(x.attrs))

def verify_science(artifacts: List[Artifact], steps: List[Step], intent: Intent):
    A=[artifact_copy(x) for x in artifacts]; findings=[]
    def add(code,verdict,message,step_index=None,evidence=None,repair=None):
        findings.append({'code':code,'verdict':verdict,'message':message,'step_index':step_index,'evidence':evidence or {},'repair':repair})
    for i,x in enumerate(A):
        if not x.provenance: add('PROVENANCE_MISSING','UNPROVEN',f'Provenance is missing for artifact {i}: {x.name}.',evidence={'artifact':i},repair='Attach authoritative product/version/acquisition metadata.')
        if x.resolution_m is not None and x.resolution_m <= 0: add('RESOLUTION_INVALID','REFUTED','Spatial resolution must be positive.',evidence={'artifact':i})
        if x.observed_min is not None and x.observed_max is not None and x.observed_min > x.observed_max: add('VALUE_RANGE_INVALID','REFUTED','Observed minimum exceeds observed maximum.',evidence={'artifact':i})
    for si,s in enumerate(steps):
        p=s.params; op=s.op
        if op=='resample':
            i=p['artifact']; method=str(p.get('method','nearest')).lower(); target=float(p['target_resolution_m']); x=A[i]
            if x.scale in CATEGORICAL_SCALES and method in CONTINUOUS_INTERPOLATORS: add('CATEGORICAL_INTERPOLATION','REFUTED',f'{method} interpolation does not preserve {x.scale} identity.',si,{'measurement_scale':x.scale,'method':method},'Use nearest-neighbour for identity preservation or mode/majority for aggregation.')
            if x.resolution_m is not None and target < x.resolution_m:
                gain=x.resolution_m/target
                if gain>1.25 and not p.get('evidence_of_new_information',False): add('UNSUPPORTED_RESOLUTION_GAIN','REFUTED',f'Grid spacing improves {gain:.2f}x without evidence of new information.',si,{'native_m':x.resolution_m,'target_m':target})
            x.resolution_m=target
        elif op=='temporal_aggregate':
            i=p['artifact']; x=A[i]; method=str(p.get('method','')).lower(); duration_h=p.get('sample_duration_h'); declared=p.get('declared_quantity')
            if x.unit in RATE_UNITS and declared=='accumulation':
                if method in {'mean','average'}: add('RATE_AVERAGED_AS_ACCUMULATION','REFUTED','A rate is averaged but the result is declared as an accumulation.',si,{'unit':x.unit,'method':method})
                elif method=='sum' and duration_h not in {1,1.0}: add('RATE_SUM_WITHOUT_DURATION','REFUTED','Sub-hourly rates are summed without multiplying by interval duration.',si,{'sample_duration_h':duration_h})
                elif method in {'integrate','duration_weighted_sum'}: x.quantity='precipitation_accumulation'; x.unit='mm'
        elif op=='area':
            i=p['artifact']; x=A[i]; method=p.get('method','planar')
            if x.crs in {'EPSG:4326','OGC:CRS84','CRS84'} and method=='planar': add('ANGULAR_COORDINATE_AREA','REFUTED','Planar area is computed directly from angular coordinates.',si,{'crs':x.crs},'Use geodesic area or an appropriate projected/equal-area CRS.')
            x.quantity='area'; x.unit=p.get('output_unit','km2')
        elif op=='spectral_index':
            red=A[p['red_artifact']]; other=A[p['other_artifact']]; expected_other=p.get('expected_other_role')
            if red.band_role!='red': add('RED_ROLE_MISMATCH','REFUTED',f'Red-band role required; got {red.band_role}.',si)
            if expected_other and other.band_role!=expected_other: add('SPECTRAL_ROLE_MISMATCH','REFUTED',f'{expected_other} role required; got {other.band_role}.',si,{'expected_role':expected_other,'actual_role':other.band_role})
            if red.resolution_m is not None and other.resolution_m is not None and red.resolution_m!=other.resolution_m and not p.get('explicit_alignment',False): add('BAND_GRID_MISMATCH','REFUTED','Spectral operands have different native resolutions without explicit alignment.',si,{'red_m':red.resolution_m,'other_m':other.resolution_m})
            if p.get('output_artifact') is not None:
                out=A[p['output_artifact']]; out.quantity=p.get('output_quantity',out.quantity); out.unit='dimensionless'
        elif op=='require_surface_reflectance':
            x=A[p['artifact']]
            if x.processing_level not in {'L2A','surface_reflectance'}: add('PROCESSING_LEVEL_MISMATCH','REFUTED',f'Surface reflectance required; got {x.processing_level}.',si,{'processing_level':x.processing_level})
        elif op=='require_cloud_mask':
            x=A[p['artifact']]
            if not bool(x.attrs.get('cloud_mask_applied',False)): add('CLOUD_MASK_MISSING','REFUTED','Cloud-sensitive analysis has no recorded cloud mask.',si)
        elif op=='require_bare_earth':
            x=A[p['artifact']]
            if x.surface_model=='DSM': add('DSM_USED_AS_DTM','REFUTED','Bare-earth analysis is requested from a DSM.',si)
            elif x.surface_model is None: add('SURFACE_MODEL_UNKNOWN','UNPROVEN','DSM/DTM semantics are not known.',si)
        elif op=='combine_vertical':
            x=A[p['a']]; y=A[p['b']]
            if not x.vertical_datum or not y.vertical_datum: add('VERTICAL_DATUM_UNKNOWN','UNPROVEN','Vertical-datum compatibility cannot be established.',si)
            elif x.vertical_datum!=y.vertical_datum: add('VERTICAL_DATUM_MISMATCH','REFUTED','Vertical quantities use incompatible datums.',si,{'a':x.vertical_datum,'b':y.vertical_datum})
        elif op=='predict':
            decision=parse_dt(intent.decision_time)
            for i in p.get('features',range(len(A))):
                t=parse_dt(A[i].available_time or A[i].acquisition_time)
                if decision is not None and t is None: add('FEATURE_TIME_UNKNOWN','UNPROVEN','Feature availability time is unknown.',si,{'artifact':i})
                elif decision is not None and t>decision: add('FUTURE_INFORMATION','REFUTED','A model feature becomes available after the decision time.',si,{'artifact':i,'feature_time':str(t),'decision_time':str(decision)})
        elif op=='split':
            mode=p.get('mode')
            if mode=='spatial':
                sep=float(p.get('separation_m',0)); corr=float(p.get('dependence_range_m',0))
                if corr>0 and sep<corr: add('SPATIAL_DEPENDENCE_LEAKAGE','REFUTED','Train/test separation is below the estimated spatial dependence range.',si,{'separation_m':sep,'dependence_range_m':corr})
            elif mode=='temporal':
                train_end=parse_dt(p.get('train_end')); test_start=parse_dt(p.get('test_start'))
                if train_end and test_start and train_end>=test_start: add('TEMPORAL_SPLIT_OVERLAP','REFUTED','Training and testing periods overlap or are not chronologically ordered.',si)
            elif mode=='scene':
                overlap=float(p.get('scene_overlap_fraction',0))
                if overlap>0: add('SCENE_OVERLAP_LEAKAGE','REFUTED','The same scene content appears in both train and test partitions.',si,{'overlap_fraction':overlap})
        elif op=='fill_nodata':
            x=A[p['artifact']]; fill=p.get('value')
            if x.nodata_semantics in {'missing','masked'} and fill==0 and p.get('zero_is_physically_valid',True): add('NODATA_COLLAPSED_TO_ZERO','REFUTED','Missing data are replaced by a physically meaningful zero.',si)
        elif op=='compare_sensors':
            x=A[p['a']]; y=A[p['b']]; tx=parse_dt(x.acquisition_time); ty=parse_dt(y.acquisition_time); tol=float(p.get('tolerance_h',intent.temporal_tolerance_h or 0))
            if not tx or not ty: add('ACQUISITION_TIME_UNKNOWN','UNPROVEN','Multisensor acquisition-time compatibility is unknown.',si)
            else:
                gap=abs((tx-ty).total_seconds())/3600
                if gap>tol: add('MULTISENSOR_TIME_MISMATCH','REFUTED','Acquisition gap exceeds the declared scientific tolerance.',si,{'gap_h':gap,'tolerance_h':tol})
        elif op=='convert_unit':
            x=A[p['artifact']]; src=x.unit; dst=p['to']; ds,dd=unit_dimension(src),unit_dimension(dst)
            if ds is None or dd is None: add('UNIT_DIMENSION_UNKNOWN','UNPROVEN','Unit dimensionality is not known.',si,{'from':src,'to':dst})
            elif ds!=dd and not ({ds,dd} <= {'length','length_or_accum'}): add('UNIT_DIMENSION_MISMATCH','REFUTED','Requested conversion changes physical dimension.',si,{'from':src,'to':dst})
            else: x.unit=dst
        elif op=='assert_value_range':
            x=A[p['artifact']]; lo,hi=p['min'],p['max']
            if x.observed_min is None or x.observed_max is None: add('VALUE_RANGE_UNOBSERVED','UNPROVEN','Required physical bounds cannot be checked because observations are missing.',si)
            elif x.observed_min<lo or x.observed_max>hi: add('PHYSICAL_RANGE_VIOLATION','REFUTED','Observed values violate declared physical bounds.',si,{'observed':[x.observed_min,x.observed_max],'required':[lo,hi]})
        elif op=='aggregate_categorical':
            x=A[p['artifact']]; method=p.get('method')
            if x.scale in CATEGORICAL_SCALES and method in {'mean','median'}: add('CATEGORICAL_AGGREGATION_INVALID','REFUTED',f'{method} is not label-preserving for nominal classes.',si)
        elif op=='drop_quality_metadata':
            if p.get('required_downstream',False): add('QUALITY_METADATA_LOST','REFUTED','A downstream-required quality flag is discarded.',si)
        elif op=='declare_uncertainty': A[p['artifact']].uncertainty=p.get('value')
    if intent.required_unit:
        result_like=[x for x in A if x.quantity==intent.required_quantity] if intent.required_quantity else A
        if result_like:
            known_units={x.unit for x in result_like if x.unit}
            if known_units and intent.required_unit not in known_units:
                req_dim=unit_dimension(intent.required_unit); actual_dims={unit_dimension(u) for u in known_units}
                if req_dim not in actual_dims: add('OUTPUT_DIMENSION_MISMATCH','REFUTED','Output physical dimension is incompatible with the scientific intent.',evidence={'required':intent.required_unit,'actual':sorted(known_units)})
    if intent.max_resolution_m is not None:
        for i,x in enumerate(A):
            if x.resolution_m is None: add('RESOLUTION_UNKNOWN','UNPROVEN','Required spatial support cannot be checked.',evidence={'artifact':i})
            elif x.resolution_m>intent.max_resolution_m: add('RESOLUTION_TOO_COARSE','REFUTED','Available spatial resolution is coarser than the declared requirement.',evidence={'artifact':i,'resolution_m':x.resolution_m,'required_m':intent.max_resolution_m})
    if intent.max_uncertainty is not None:
        known=[x.uncertainty for x in A if x.uncertainty is not None]
        if not known: add('UNCERTAINTY_UNKNOWN','UNPROVEN','The intent defines an uncertainty limit but no uncertainty evidence is available.')
        elif max(known)>intent.max_uncertainty: add('UNCERTAINTY_EXCEEDS_LIMIT','REFUTED','Output uncertainty exceeds the declared fitness-for-purpose limit.',evidence={'max_observed':max(known),'limit':intent.max_uncertainty})
    if intent.required_value_range:
        lo,hi=intent.required_value_range; relevant=[x for x in A if x.quantity==intent.required_quantity] if intent.required_quantity else A
        for x in relevant:
            if x.observed_min is None or x.observed_max is None: add('INTENT_RANGE_UNPROVEN','UNPROVEN','Intent-specific physical bounds cannot be checked.')
            elif x.observed_min<lo or x.observed_max>hi: add('INTENT_RANGE_REFUTED','REFUTED','Output violates the physical range required by the declared scientific intent.')
    verdict='REFUTED' if any(f['verdict']=='REFUTED' for f in findings) else ('UNPROVEN' if any(f['verdict']=='UNPROVEN' for f in findings) else 'PROVEN')
    return {'verdict':verdict,'findings':findings,'final_artifacts':[asdict(x) for x in A]}

# -------- New evidence-carrying certificate layer --------
OBLIGATION_LIBRARY = {
'CATEGORICAL_INTERPOLATION': ('R-TYPE-MEASUREMENT-SCALE', 'Continuous interpolation requires a continuous measurement scale.', ['measurement_scale','method']),
'UNSUPPORTED_RESOLUTION_GAIN': ('R-SPATIAL-SUPPORT', 'A finer grid is not new observational support unless new information is evidenced.', ['native_m','target_m']),
'RATE_AVERAGED_AS_ACCUMULATION': ('R-TEMPORAL-QUANTITY', 'A rate must be integrated over time before it can be interpreted as an accumulation.', ['unit','method']),
'RATE_SUM_WITHOUT_DURATION': ('R-TEMPORAL-QUANTITY', 'Sub-hourly rates require duration-weighting for accumulation.', ['sample_duration_h']),
'RED_ROLE_MISMATCH': ('R-SPECTRAL-ROLE', 'The red operand of the requested index must have red-band semantics.', []),
'SPECTRAL_ROLE_MISMATCH': ('R-SPECTRAL-ROLE', 'The non-red operand must match the spectral role required by the index.', ['expected_role','actual_role']),
'BAND_GRID_MISMATCH': ('R-SPATIAL-ALIGNMENT', 'Spectral operands must share support or have explicit alignment.', ['red_m','other_m']),
'PROCESSING_LEVEL_MISMATCH': ('R-PROCESSING-LEVEL', 'The workflow requires surface reflectance semantics.', ['processing_level']),
'CLOUD_MASK_MISSING': ('R-QUALITY-MASK', 'Cloud-sensitive analysis requires recorded cloud handling.', []),
'DSM_USED_AS_DTM': ('R-VERTICAL-SURFACE', 'Bare-earth claims require DTM-like rather than DSM-like surface semantics.', []),
'SURFACE_MODEL_UNKNOWN': ('R-VERTICAL-SURFACE', 'Surface-model semantics are required to decide a bare-earth claim.', []),
'VERTICAL_DATUM_UNKNOWN': ('R-VERTICAL-DATUM', 'Vertical compatibility requires both vertical datums.', []),
'VERTICAL_DATUM_MISMATCH': ('R-VERTICAL-DATUM', 'Combined vertical quantities require compatible datums or an explicit transformation.', ['a','b']),
'FEATURE_TIME_UNKNOWN': ('R-CAUSALITY', 'Prediction features require known availability time relative to the decision time.', ['artifact']),
'FUTURE_INFORMATION': ('R-CAUSALITY', 'A predictive workflow must not use information unavailable at decision time.', ['feature_time','decision_time']),
'SPATIAL_DEPENDENCE_LEAKAGE': ('R-EVAL-INDEPENDENCE', 'Train/test separation must exceed the declared spatial dependence range.', ['separation_m','dependence_range_m']),
'TEMPORAL_SPLIT_OVERLAP': ('R-EVAL-INDEPENDENCE', 'Training and testing periods must be chronologically independent.', []),
'SCENE_OVERLAP_LEAKAGE': ('R-EVAL-INDEPENDENCE', 'Scene content must not overlap between train and test partitions.', ['overlap_fraction']),
'NODATA_COLLAPSED_TO_ZERO': ('R-NODATA-SEMANTICS', 'Missingness must not be collapsed into a physically meaningful zero.', []),
'MULTISENSOR_TIME_MISMATCH': ('R-TEMPORAL-SUPPORT', 'Multisensor observations must meet the declared acquisition-time tolerance.', ['gap_h','tolerance_h']),
'ACQUISITION_TIME_UNKNOWN': ('R-TEMPORAL-SUPPORT', 'Acquisition times are required to establish multisensor compatibility.', []),
'UNIT_DIMENSION_UNKNOWN': ('R-DIMENSION', 'Unit dimensions must be known before conversion.', ['from','to']),
'UNIT_DIMENSION_MISMATCH': ('R-DIMENSION', 'Unit conversion must preserve physical dimension.', ['from','to']),
'PHYSICAL_RANGE_VIOLATION': ('R-PHYSICAL-DOMAIN', 'Observed values must satisfy declared physical bounds.', ['observed','required']),
'VALUE_RANGE_UNOBSERVED': ('R-PHYSICAL-DOMAIN', 'Observed bounds are required to check a physical-domain claim.', []),
'CATEGORICAL_AGGREGATION_INVALID': ('R-TYPE-MEASUREMENT-SCALE', 'Nominal classes require label-preserving aggregation.', []),
'ANGULAR_COORDINATE_AREA': ('R-GEOMETRY', 'Area over geographic coordinates requires geodesic or suitable projected geometry.', ['crs']),
'QUALITY_METADATA_LOST': ('R-PROVENANCE-QUALITY', 'Quality evidence required downstream must be preserved.', []),
'PROVENANCE_MISSING': ('R-PROVENANCE', 'Authoritative source provenance is required for scientific auditability.', ['artifact']),
'RESOLUTION_INVALID': ('R-SPATIAL-SUPPORT', 'Spatial resolution must be positive.', ['artifact']),
'VALUE_RANGE_INVALID': ('R-PHYSICAL-DOMAIN', 'Observed minimum cannot exceed observed maximum.', ['artifact']),
'OUTPUT_DIMENSION_MISMATCH': ('R-INTENT-DIMENSION', 'Output dimension must satisfy the scientific intent.', ['required','actual']),
'RESOLUTION_UNKNOWN': ('R-INTENT-SPATIAL', 'Spatial support must be known for an intent with a resolution requirement.', ['artifact']),
'RESOLUTION_TOO_COARSE': ('R-INTENT-SPATIAL', 'Available support must meet the maximum resolution required by the intent.', ['resolution_m','required_m']),
'UNCERTAINTY_UNKNOWN': ('R-INTENT-UNCERTAINTY', 'Uncertainty evidence is required when the intent declares a limit.', []),
'UNCERTAINTY_EXCEEDS_LIMIT': ('R-INTENT-UNCERTAINTY', 'Uncertainty must remain within the declared fitness-for-purpose limit.', ['max_observed','limit']),
'INTENT_RANGE_UNPROVEN': ('R-INTENT-DOMAIN', 'Intent-specific physical bounds require observed output bounds.', []),
'INTENT_RANGE_REFUTED': ('R-INTENT-DOMAIN', 'Output values must satisfy the physical range required by the intent.', []),
}

REPAIRABLE_CODES = {
'CATEGORICAL_INTERPOLATION','UNSUPPORTED_RESOLUTION_GAIN','RATE_AVERAGED_AS_ACCUMULATION','RATE_SUM_WITHOUT_DURATION',
'RED_ROLE_MISMATCH','SPECTRAL_ROLE_MISMATCH','PROCESSING_LEVEL_MISMATCH','CLOUD_MASK_MISSING','DSM_USED_AS_DTM',
'FUTURE_INFORMATION','SPATIAL_DEPENDENCE_LEAKAGE','TEMPORAL_SPLIT_OVERLAP','SCENE_OVERLAP_LEAKAGE','NODATA_COLLAPSED_TO_ZERO',
'CATEGORICAL_AGGREGATION_INVALID','ANGULAR_COORDINATE_AREA'
}

def _fingerprint(artifacts, steps, intent):
    payload={'artifacts':[asdict(x) for x in artifacts],'steps':[asdict(x) for x in steps],'intent':asdict(intent)}
    return hashlib.sha256(json.dumps(payload,sort_keys=True,default=str).encode()).hexdigest()

def evidence_certificate(artifacts, steps, intent, result=None, case_id=None):
    result=result or verify_science(artifacts,steps,intent)
    obs=[]
    for f in result['findings']:
        oid,pred,req=OBLIGATION_LIBRARY.get(f['code'],('R-UNCLASSIFIED','Workflow must satisfy the encoded scientific constraint.',[]))
        missing=[k for k in req if k not in (f.get('evidence') or {}) or (f.get('evidence') or {}).get(k) is None]
        obs.append({
            'obligation_id':oid,'code':f['code'],'status':f['verdict'],'predicate':pred,
            'step_index':f.get('step_index'),'evidence':f.get('evidence') or {},'missing_evidence':missing,
            'message':f['message'],'suggested_repair':f.get('repair'),'automatically_repairable':f['code'] in REPAIRABLE_CODES
        })
    cert={'schema':'SciGeoGuard-X/EvidenceCertificate-v0.1','case_id':case_id,'verdict':result['verdict'],
          'workflow_hash':_fingerprint(artifacts,steps,intent),'n_obligations_reported':len(obs),'obligations':obs}
    return cert

def _candidate(artifacts, steps, intent, description, mutate, source_code):
    a=deepcopy(artifacts); s=deepcopy(steps); it=deepcopy(intent); mutate(a,s,it)
    return {'artifacts':a,'steps':s,'intent':it,'description':description,'source_code':source_code,'cost':1}

def repair_candidates(artifacts, steps, intent, result=None):
    result=result or verify_science(artifacts,steps,intent); out=[]
    for f in result['findings']:
        if f['verdict']!='REFUTED': continue
        code=f['code']; si=f.get('step_index'); ev=f.get('evidence') or {}
        if code=='CATEGORICAL_INTERPOLATION' and si is not None:
            out.append(_candidate(artifacts,steps,intent,'Replace continuous interpolation with nearest-neighbour.',lambda a,s,i,si=si:s[si].params.__setitem__('method','nearest'),code))
            out.append(_candidate(artifacts,steps,intent,'Replace continuous interpolation with mode aggregation.',lambda a,s,i,si=si:s[si].params.__setitem__('method','mode'),code))
        elif code=='UNSUPPORTED_RESOLUTION_GAIN' and si is not None:
            native=ev.get('native_m')
            if native is not None: out.append(_candidate(artifacts,steps,intent,'Keep grid spacing at the native support; do not invent finer observational support.',lambda a,s,i,si=si,n=native:s[si].params.__setitem__('target_resolution_m',float(n)),code))
        elif code in {'RATE_AVERAGED_AS_ACCUMULATION','RATE_SUM_WITHOUT_DURATION'} and si is not None:
            idx=steps[si].params.get('artifact'); duration=artifacts[idx].temporal_resolution_h if idx is not None else None
            def mut(a,s,i,si=si,d=duration):
                s[si].params['method']='duration_weighted_sum'
                if d is not None: s[si].params['sample_duration_h']=d
            out.append(_candidate(artifacts,steps,intent,'Integrate the rate using the sample duration.',mut,code))
        elif code=='RED_ROLE_MISMATCH' and si is not None:
            for j,x in enumerate(artifacts):
                if x.band_role=='red': out.append(_candidate(artifacts,steps,intent,f'Use artifact {j} ({x.name}) as the red operand.',lambda a,s,i,si=si,j=j:s[si].params.__setitem__('red_artifact',j),code))
        elif code=='SPECTRAL_ROLE_MISMATCH' and si is not None:
            expected=steps[si].params.get('expected_other_role')
            for j,x in enumerate(artifacts):
                if expected and x.band_role==expected: out.append(_candidate(artifacts,steps,intent,f'Use artifact {j} ({x.name}) for spectral role {expected}.',lambda a,s,i,si=si,j=j:s[si].params.__setitem__('other_artifact',j),code))
        elif code=='PROCESSING_LEVEL_MISMATCH' and si is not None:
            for j,x in enumerate(artifacts):
                if x.processing_level in {'L2A','surface_reflectance'}: out.append(_candidate(artifacts,steps,intent,f'Use surface-reflectance artifact {j} ({x.name}).',lambda a,s,i,si=si,j=j:s[si].params.__setitem__('artifact',j),code))
        elif code=='CLOUD_MASK_MISSING' and si is not None:
            for j,x in enumerate(artifacts):
                if bool(x.attrs.get('cloud_mask_applied',False)): out.append(_candidate(artifacts,steps,intent,f'Use artifact {j} ({x.name}) with recorded cloud masking.',lambda a,s,i,si=si,j=j:s[si].params.__setitem__('artifact',j),code))
        elif code=='DSM_USED_AS_DTM' and si is not None:
            for j,x in enumerate(artifacts):
                if x.surface_model=='DTM': out.append(_candidate(artifacts,steps,intent,f'Use bare-earth artifact {j} ({x.name}).',lambda a,s,i,si=si,j=j:s[si].params.__setitem__('artifact',j),code))
        elif code=='FUTURE_INFORMATION' and si is not None:
            bad=ev.get('artifact'); features=list(steps[si].params.get('features',range(len(artifacts))))
            if bad in features and len(features)>1:
                new=[x for x in features if x!=bad]
                out.append(_candidate(artifacts,steps,intent,f'Remove feature {bad} that is unavailable at decision time.',lambda a,s,i,si=si,new=new:s[si].params.__setitem__('features',new),code))
        elif code=='SPATIAL_DEPENDENCE_LEAKAGE' and si is not None:
            dep=ev.get('dependence_range_m')
            if dep is not None: out.append(_candidate(artifacts,steps,intent,'Increase train/test spatial separation to the dependence range.',lambda a,s,i,si=si,d=dep:s[si].params.__setitem__('separation_m',float(d)),code))
        elif code=='TEMPORAL_SPLIT_OVERLAP' and si is not None:
            te=parse_dt(steps[si].params.get('train_end'))
            if te:
                ts=(te+timedelta(seconds=1)).isoformat()
                out.append(_candidate(artifacts,steps,intent,'Move the test period to begin after the training period.',lambda a,s,i,si=si,ts=ts:s[si].params.__setitem__('test_start',ts),code))
        elif code=='SCENE_OVERLAP_LEAKAGE' and si is not None:
            out.append(_candidate(artifacts,steps,intent,'Repartition scenes to eliminate train/test content overlap.',lambda a,s,i,si=si:s[si].params.__setitem__('scene_overlap_fraction',0.0),code))
        elif code=='NODATA_COLLAPSED_TO_ZERO' and si is not None:
            out.append(_candidate(artifacts,steps,intent,'Preserve missingness instead of filling with physical zero.',lambda a,s,i,si=si:s[si].params.__setitem__('value',None),code))
        elif code=='CATEGORICAL_AGGREGATION_INVALID' and si is not None:
            out.append(_candidate(artifacts,steps,intent,'Use mode aggregation for nominal classes.',lambda a,s,i,si=si:s[si].params.__setitem__('method','mode'),code))
        elif code=='ANGULAR_COORDINATE_AREA' and si is not None:
            out.append(_candidate(artifacts,steps,intent,'Use geodesic area for geographic coordinates.',lambda a,s,i,si=si:s[si].params.__setitem__('method','geodesic'),code))
    # Deduplicate by workflow fingerprint
    seen=set(); uniq=[]
    for c in out:
        fp=_fingerprint(c['artifacts'],c['steps'],c['intent'])
        if fp not in seen: seen.add(fp); uniq.append(c)
    return uniq

def synthesize_minimal_repair(artifacts, steps, intent, max_edits=2, max_states=500):
    initial=verify_science(artifacts,steps,intent)
    if initial['verdict']=='PROVEN': return {'status':'ALREADY_PROVEN','cost':0,'edits':[],'result':initial,'artifacts':deepcopy(artifacts),'steps':deepcopy(steps),'intent':deepcopy(intent)}
    q=deque([(deepcopy(artifacts),deepcopy(steps),deepcopy(intent),[],0)]); seen={_fingerprint(artifacts,steps,intent)}; explored=0
    while q and explored<max_states:
        a,s,it,edits,cost=q.popleft(); explored+=1
        r=verify_science(a,s,it)
        if r['verdict']=='PROVEN' and cost>0:
            return {'status':'REPAIRED','cost':cost,'edits':edits,'result':r,'artifacts':a,'steps':s,'intent':it,'states_explored':explored}
        if cost>=max_edits: continue
        for cand in repair_candidates(a,s,it,r):
            fp=_fingerprint(cand['artifacts'],cand['steps'],cand['intent'])
            if fp in seen: continue
            seen.add(fp); q.append((cand['artifacts'],cand['steps'],cand['intent'],edits+[cand['description']],cost+1))
    return {'status':'NO_SAFE_REPAIR_FOUND','cost':None,'edits':[],'result':initial,'states_explored':explored}

def make_metamorphic_suite():
    suite=[]
    # 1 categorical resampling
    a=[Artifact('WorldCover','land_cover',scale='categorical',resolution_m=10,provenance='ESA WorldCover')]
    valid=[Step('resample',{'artifact':0,'method':'nearest','target_resolution_m':20})]; mut=deepcopy(valid); mut[0].params['method']='bilinear'
    suite.append(('categorical_interpolation',a,valid,mut,Intent('preserve class identity')))
    # 2 spectral role
    a=[Artifact('B04','reflectance',band_role='red',resolution_m=10,processing_level='L2A',provenance='Sentinel-2 L2A'),Artifact('B08','reflectance',band_role='nir',resolution_m=10,processing_level='L2A',provenance='Sentinel-2 L2A'),Artifact('B11','reflectance',band_role='swir',resolution_m=20,processing_level='L2A',provenance='Sentinel-2 L2A')]
    valid=[Step('spectral_index',{'red_artifact':0,'other_artifact':1,'expected_other_role':'nir','explicit_alignment':True})]; mut=deepcopy(valid); mut[0].params['other_artifact']=2
    suite.append(('spectral_role_substitution',a,valid,mut,Intent('NDVI')))
    # 3 temporal quantity
    a=[Artifact('IMERG','precipitation_rate',unit='MM/Hr',temporal_resolution_h=0.5,provenance='GPM IMERG V07')]
    valid=[Step('temporal_aggregate',{'artifact':0,'method':'duration_weighted_sum','sample_duration_h':0.5,'declared_quantity':'accumulation'})]; mut=deepcopy(valid); mut[0].params['method']='sum'
    suite.append(('rate_to_accumulation',a,valid,mut,Intent('daily precipitation accumulation')))
    # 4 spatial support
    a=[Artifact('CopDEM','elevation',unit='m',scale='continuous',resolution_m=30,provenance='Copernicus DEM GLO-30')]
    valid=[Step('resample',{'artifact':0,'method':'bilinear','target_resolution_m':30})]; mut=deepcopy(valid); mut[0].params['target_resolution_m']=10
    suite.append(('unsupported_resolution_gain',a,valid,mut,Intent('terrain model')))
    # 5 causality
    a=[Artifact('past','feature',available_time='2024-06-01T00:00:00+00:00',provenance='source-A'),Artifact('future','feature',available_time='2024-07-10T00:00:00+00:00',provenance='source-B')]
    valid=[Step('predict',{'features':[0]})]; mut=deepcopy(valid); mut[0].params['features']=[0,1]
    suite.append(('future_information',a,valid,mut,Intent('forecast',decision_time='2024-07-01T00:00:00+00:00')))
    # 6 bare-earth semantics
    a=[Artifact('DTM','elevation',surface_model='DTM',provenance='terrain-product'),Artifact('DSM','elevation',surface_model='DSM',provenance='surface-product')]
    valid=[Step('require_bare_earth',{'artifact':0})]; mut=deepcopy(valid); mut[0].params['artifact']=1
    suite.append(('dsm_as_dtm',a,valid,mut,Intent('bare-earth slope')))
    # 7 spatial evaluation leakage
    a=[Artifact('samples','image_samples',provenance='dataset')]
    valid=[Step('split',{'mode':'spatial','separation_m':100,'dependence_range_m':50})]; mut=deepcopy(valid); mut[0].params['separation_m']=10
    suite.append(('spatial_leakage',a,valid,mut,Intent('independent evaluation')))
    # 8 categorical aggregation
    a=[Artifact('classes','land_cover',scale='nominal',provenance='map')]
    valid=[Step('aggregate_categorical',{'artifact':0,'method':'mode'})]; mut=deepcopy(valid); mut[0].params['method']='mean'
    suite.append(('categorical_aggregation',a,valid,mut,Intent('dominant class')))
    # 9 processing level
    a=[Artifact('L2A','reflectance',processing_level='L2A',provenance='S2'),Artifact('L1C','radiance_like',processing_level='L1C',provenance='S2')]
    valid=[Step('require_surface_reflectance',{'artifact':0})]; mut=deepcopy(valid); mut[0].params['artifact']=1
    suite.append(('processing_level',a,valid,mut,Intent('surface-reflectance analysis')))
    # 10 vertical datum mismatch - deliberately not auto-repaired because inventing a datum conversion is unsafe
    a=[Artifact('terrain','elevation',unit='m',vertical_datum='EGM2008',provenance='terrain'),Artifact('water','water_level',unit='m',vertical_datum='EGM2008',provenance='hydraulic')]
    valid=[Step('combine_vertical',{'a':0,'b':1})]; mut_a=deepcopy(a); mut_a[1].vertical_datum='EGM96'
    suite.append(('vertical_datum_mismatch',(a,mut_a),valid,valid,Intent('flood depth')))
    return suite

def run_metamorphic_suite():
    rows=[]; certs=[]
    for name,a,valid,mut,intent in make_metamorphic_suite():
        if name=='vertical_datum_mismatch': valid_a,mut_a=a; a_valid=valid_a; a_mut=mut_a
        else: a_valid=a_mut=a
        vr=verify_science(a_valid,valid,intent); mr=verify_science(a_mut,mut,intent); rep=synthesize_minimal_repair(a_mut,mut,intent,max_edits=2)
        rows.append({'family':name,'valid_verdict':vr['verdict'],'mutated_verdict':mr['verdict'],'detected':mr['verdict']=='REFUTED','repair_status':rep['status'],'repair_cost':rep.get('cost'),'reverified_proven':rep.get('result',{}).get('verdict')=='PROVEN' if rep['status']=='REPAIRED' else False,'first_code':mr['findings'][0]['code'] if mr['findings'] else None})
        certs.append(evidence_certificate(a_mut,mut,intent,mr,case_id=name))
    return rows,certs


## 2. Single counterexample certificate
First test a categorical land-cover workflow. The original operation uses bilinear interpolation on nominal class identifiers. The verifier should refute the workflow and the certificate should identify the measurement-scale obligation.


In [ ]:
artifacts = [Artifact(
    name='WorldCover', quantity='land_cover', scale='categorical',
    resolution_m=10, provenance='ESA WorldCover 2021 v200'
)]
steps = [Step('resample', {'artifact':0, 'method':'bilinear', 'target_resolution_m':20})]
intent = Intent(goal='preserve land-cover class identity')

result = verify_science(artifacts, steps, intent)
certificate = evidence_certificate(artifacts, steps, intent, result, case_id='DEMO_WORLDCOVER')
print('verdict =', result['verdict'])
print(json.dumps(certificate, indent=2))
assert result['verdict'] == 'REFUTED'
assert certificate['obligations'][0]['obligation_id'] == 'R-TYPE-MEASUREMENT-SCALE'


## 3. Minimal repair synthesis and re-verification
The search explores only edits generated from the current counterexample. For the land-cover example it should prefer a one-edit label-preserving resampling method and then re-run the verifier.


In [ ]:
repair = synthesize_minimal_repair(artifacts, steps, intent, max_edits=2)
print('repair status =', repair['status'])
print('repair cost   =', repair['cost'])
print('edits         =', repair['edits'])
print('new verdict   =', repair['result']['verdict'])
print('new step      =', asdict(repair['steps'][0]))
assert repair['status'] == 'REPAIRED'
assert repair['cost'] == 1
assert repair['result']['verdict'] == 'PROVEN'


## 4. Scientific metamorphic test suite
Each row starts from a workflow that should be `PROVEN`, applies one scientifically meaningful mutation, and checks three properties:

\[
V(W)=	exttt{PROVEN},\qquad V(\mu(W))=	exttt{REFUTED},\qquad V(R(\mu(W)))=	exttt{PROVEN}
\]

where the last condition is evaluated only when the repair library can make a safe edit. The vertical-datum mutation is intentionally expected to be **detected but not automatically repaired**, because changing a datum requires an actual transformation, not a metadata edit.


In [ ]:
rows, certificates = run_metamorphic_suite()
metamorphic = pd.DataFrame(rows)
metamorphic.to_csv(RUN_DIR/'v11_metamorphic_results.csv', index=False)
(RUN_DIR/'v11_metamorphic_certificates.json').write_text(json.dumps(certificates, indent=2))
display(metamorphic)

summary = {
    'n_families': int(len(metamorphic)),
    'valid_seed_proven_rate': float((metamorphic.valid_verdict=='PROVEN').mean()),
    'mutation_detection_rate': float(metamorphic.detected.mean()),
    'safe_repair_found_rate': float((metamorphic.repair_status=='REPAIRED').mean()),
    'repair_reverification_rate_among_repaired': float(metamorphic.loc[metamorphic.repair_status=='REPAIRED','reverified_proven'].mean()),
    'mean_edit_cost_among_repaired': float(metamorphic.loc[metamorphic.repair_status=='REPAIRED','repair_cost'].mean()),
}
print(json.dumps(summary, indent=2))
assert summary['valid_seed_proven_rate'] == 1.0
assert summary['mutation_detection_rate'] == 1.0
assert summary['repair_reverification_rate_among_repaired'] == 1.0


### Interpreting this result
A 100% mutation-detection rate here is **not** a generalization result: these are controlled metamorphic tests built around encoded invariants. The useful quantities are (a) whether the certificate localizes the right scientific obligation, (b) whether repair is conservative, and (c) whether repaired workflows are re-verified rather than accepted without checking.


## 5. Frozen 180-case benchmark integration
The package includes the exact v6 `Blinded_180_cases.csv` and `Frozen_v3_reference.csv` snapshots. If they are not found locally, the cell falls back to the public GitHub raw files.

We first confirm that the frozen verifier still reproduces the reference. We then attempt **safe repair only on refuted cases**. This answers a new question the earlier paper did not study: *how much of the benchmark is automatically repairable under a conservative repair policy?*


In [ ]:
def locate_or_download(name):
    candidates = [
        Path('data')/name,
        Path.cwd()/name,
        Path('/content')/name,
    ]
    for p in candidates:
        if p.exists():
            return p
    url = f'https://raw.githubusercontent.com/bnssaanirudh/SciGeoGuard/main/results/history/v6/{name}'
    print('Downloading', url)
    import urllib.request
    target = RUN_DIR/name
    urllib.request.urlretrieve(url, target)
    return target

blind_path = locate_or_download('Blinded_180_cases.csv')
ref_path   = locate_or_download('Frozen_v3_reference.csv')
blind = pd.read_csv(blind_path)
ref = pd.read_csv(ref_path)
bench = blind.merge(ref[['case_id','generated_expected']], on='case_id', how='inner')
print('cases =', len(bench))
assert len(bench) == 180


In [ ]:
benchmark_rows = []
certificate_rows = []
for _, row in bench.iterrows():
    artifacts = [Artifact(**x) for x in json.loads(row.artifacts_json)]
    steps = [Step(**x) for x in json.loads(row.steps_json)]
    intent = Intent(**json.loads(row.workflow_intent))
    vr = verify_science(artifacts, steps, intent)
    cert = evidence_certificate(artifacts, steps, intent, vr, case_id=row.case_id)
    rep = synthesize_minimal_repair(artifacts, steps, intent, max_edits=2, max_states=200) if vr['verdict']=='REFUTED' else None
    benchmark_rows.append({
        'case_id': row.case_id,
        'generated_expected': row.generated_expected,
        'verdict': vr['verdict'],
        'matches_reference': vr['verdict']==row.generated_expected,
        'n_findings': len(vr['findings']),
        'first_code': vr['findings'][0]['code'] if vr['findings'] else None,
        'repair_status': rep['status'] if rep else '',
        'repair_cost': rep.get('cost') if rep else np.nan,
        'repair_reverified_proven': bool(rep and rep['status']=='REPAIRED' and rep['result']['verdict']=='PROVEN'),
        'repair_edits': json.dumps(rep.get('edits',[])) if rep else '[]',
    })
    for o in cert['obligations']:
        certificate_rows.append({'case_id':row.case_id, **o})

bench_results = pd.DataFrame(benchmark_rows)
obligation_results = pd.DataFrame(certificate_rows)
bench_results.to_csv(RUN_DIR/'v11_blind180_repair_results.csv', index=False)
obligation_results.to_json(RUN_DIR/'v11_blind180_obligations.jsonl', orient='records', lines=True)

display(bench_results.head(10))
print('verdict counts:', bench_results.verdict.value_counts().to_dict())
print('reference agreement:', bench_results.matches_reference.mean())
assert bench_results.matches_reference.all()


In [ ]:
refuted = bench_results[bench_results.verdict=='REFUTED'].copy()
repairable = refuted[refuted.repair_status=='REPAIRED'].copy()
benchmark_summary = {
    'n_cases': int(len(bench_results)),
    'n_refuted': int(len(refuted)),
    'frozen_reference_agreement': float(bench_results.matches_reference.mean()),
    'safe_repairs_found': int(len(repairable)),
    'safe_repair_rate_among_refuted': float(len(repairable)/len(refuted)),
    'reverified_proven_rate_among_found_repairs': float(repairable.repair_reverified_proven.mean()) if len(repairable) else None,
    'mean_edit_cost_among_found_repairs': float(repairable.repair_cost.mean()) if len(repairable) else None,
    'unique_obligation_families_in_findings': int(obligation_results.obligation_id.nunique()),
    'unique_finding_codes': int(obligation_results.code.nunique()),
}
(RUN_DIR/'v11_benchmark_summary.json').write_text(json.dumps(benchmark_summary, indent=2))
print(json.dumps(benchmark_summary, indent=2))

repair_by_code = (refuted.groupby('first_code')
                  .agg(n=('case_id','size'), repaired=('repair_reverified_proven','sum'))
                  .assign(repair_rate=lambda d:d.repaired/d.n)
                  .sort_values(['n','repair_rate'], ascending=[False,False]))
repair_by_code.to_csv(RUN_DIR/'v11_repair_by_first_failure.csv')
display(repair_by_code)


## 6. Why some workflows remain unrepaired
A useful repair system must be able to **refuse unsafe repairs**. Examples:

- `VERTICAL_DATUM_MISMATCH`: changing a datum label is not a vertical transformation.
- `PROVENANCE_MISSING`: provenance cannot be synthesized.
- `CLOUD_MASK_MISSING`: setting a Boolean flag is not the same as actually masking clouds.
- `MULTISENSOR_TIME_MISMATCH`: widening the tolerance would change the scientific requirement rather than repair the workflow.
- `PHYSICAL_RANGE_VIOLATION`: observed values should not be overwritten to satisfy a bound.

This distinction is important for novelty: the repair objective is **minimal scientifically admissible change**, not simply finding any edit that makes the rule checker pass.


## 7. Optional Z3 encoding of a refinement obligation
This cell is optional. In Google Colab you can install `z3-solver` and use an SMT solver to show that the categorical-interpolation case is inconsistent with the refinement predicate.

The core notebook results do not depend on this cell. If Z3 is unavailable, the cell prints installation instructions and continues.


In [ ]:
try:
    import z3
    Z3_AVAILABLE = True
except Exception:
    Z3_AVAILABLE = False

if not Z3_AVAILABLE:
    print('Z3 is not installed. In Colab run:  %pip -q install z3-solver  and then rerun this cell.')
else:
    is_continuous = z3.Bool('is_continuous')
    uses_continuous_interpolator = z3.Bool('uses_continuous_interpolator')
    valid = z3.Or(is_continuous, z3.Not(uses_continuous_interpolator))
    s = z3.Solver()
    # Known facts: WorldCover classes are nominal, bilinear is a continuous interpolator.
    s.add(is_continuous == False)
    s.add(uses_continuous_interpolator == True)
    s.add(valid)
    print('Can the workflow satisfy the refinement predicate?', s.check())
    assert s.check() == z3.unsat


## 8. Result figures
These plots are intended for experiment inspection, not automatically for the final paper. Keep the evidence boundary in the caption if you use them.


In [ ]:
fig = plt.figure(figsize=(8,4.6))
ax = fig.add_axes([0.1,0.18,0.86,0.72])
vals = [summary['mutation_detection_rate']*100,
        summary['safe_repair_found_rate']*100,
        benchmark_summary['safe_repair_rate_among_refuted']*100]
labels = ['Metamorphic\nmutation detected', 'Metamorphic\nsafe repair found', 'Blind-180 refuted cases\nsafely repaired']
bars = ax.bar(labels, vals)
ax.set_ylim(0,105); ax.set_ylabel('Percent (%)'); ax.set_title('SciGeoGuard-X v11: detection and conservative repair')
ax.grid(axis='y', alpha=0.2)
for b,v in zip(bars, vals): ax.text(b.get_x()+b.get_width()/2, v+1, f'{v:.1f}%', ha='center')
fig.savefig(RUN_DIR/'v11_repair_summary.png', dpi=220, bbox_inches='tight')
plt.show()

fig = plt.figure(figsize=(8,4.8))
ax = fig.add_axes([0.1,0.22,0.86,0.68])
top = repair_by_code.head(10).sort_values('n')
y = np.arange(len(top))
ax.barh(y, top['n'], label='refuted cases')
ax.barh(y, top['repaired'], label='safely repaired')
ax.set_yticks(y, top.index); ax.set_xlabel('Cases'); ax.set_title('Repair coverage by first failing obligation')
ax.legend(); ax.grid(axis='x', alpha=0.2)
fig.savefig(RUN_DIR/'v11_repair_by_failure.png', dpi=220, bbox_inches='tight')
plt.show()


## 9. Manuscript-safe interpretation
If your rerun reproduces the saved outputs, the **safe** claim is:

> On ten controlled scientific-semantic mutations spanning measurement scale, spectral role, temporal quantity, spatial support, causality, surface semantics, evaluation leakage, categorical aggregation, processing level, and vertical datum, all ten mutations were refuted. Nine admitted a one-edit repair that was re-verified as `PROVEN`; the vertical-datum mismatch was deliberately left unrepaired because a valid fix requires an actual datum transformation. On the frozen 180-case generated-reference subset, conservative repair synthesis found a one-edit repair for a subset of refuted workflows; this is regression-level repair coverage, not independent accuracy evidence.

Do **not** promote these results as human-validated until the independent reviewer study is completed.


## 10. Export all v11 outputs
This produces a ZIP containing CSV/JSON results, certificates and figures. In Colab it also exposes the path you can download.


In [ ]:
manifest = {
    'experiment': 'SciGeoGuard-X v11 EarthRefine / counterexample-guided repair',
    'seed': SEED,
    'metamorphic_summary': summary,
    'benchmark_summary': benchmark_summary,
    'evidence_boundary': 'Generated-reference and author-controlled metamorphic evidence; not independent human ground truth.'
}
(RUN_DIR/'v11_manifest.json').write_text(json.dumps(manifest, indent=2))
zip_path = shutil.make_archive(str(RUN_DIR), 'zip', root_dir=RUN_DIR)
print('RESULT ZIP:', zip_path)
print('Files:')
for p in sorted(RUN_DIR.iterdir()): print(' -', p.name)

try:
    from google.colab import files
    print('Colab detected. Uncomment the next line if you want an immediate browser download:')
    print("# files.download(zip_path)")
except Exception:
    pass
